In [5]:
from pathlib import Path
import numpy as np
import torch

v5_path = Path("./sentinel1-lulc/patches_v5/train_v5.npz")
v4_val_path = Path("./sentinel1-lulc/patches_v4/val_v4.npz")
v4_test_path = Path("./sentinel1-lulc/patches_v4/test_v4.npz")

assert v5_path.exists()
assert v4_val_path.exists()
assert v4_test_path.exists()

print("PYTORCH:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2
    ))

# Load V5 labels and metadata only.
with np.load(v5_path, allow_pickle=False) as data:
    y = data["y"]
    metadata = data["metadata"]

    assert y.shape == (8280, 128, 128)

    print("\nV5 training labels:", y.shape)

    print("\nCLASS PIXEL DISTRIBUTION")
    for class_id, name in {
        1: "Tree",
        2: "Grass",
        3: "Cropland",
        4: "Built-up",
        5: "Bare/sparse",
        6: "Water",
        255: "Ignore"
    }.items():
        count = int(np.count_nonzero(y == class_id))
        print(f"{name:<12}: {count:,}")

    # Calculate V4-compatible patch sampler weights.
    bare_counts = np.count_nonzero(y == 5, axis=(1, 2))
    water_counts = np.count_nonzero(y == 6, axis=(1, 2))

    rare_patch = (bare_counts >= 100) | (water_counts >= 100)

    sample_weights = np.where(rare_patch, 2.0, 1.0)

    print("\nSAMPLER CHECK")
    print("Total observations:", len(sample_weights))
    print("Weight 1.0:", int(np.count_nonzero(sample_weights == 1.0)))
    print("Weight 2.0:", int(np.count_nonzero(sample_weights == 2.0)))
    print("Bare-eligible patches:", int(np.count_nonzero(bare_counts >= 100)))
    print("Water-eligible patches:", int(np.count_nonzero(water_counts >= 100)))

# Confirm validation and test files remain present and unchanged in shape.
for name, path in [("Validation", v4_val_path), ("Test", v4_test_path)]:
    with np.load(path, allow_pickle=False) as data:
        print(f"\n{name}: X={data['X'].shape}, y={data['y'].shape}")

print("\nV5 TRAINING PREFLIGHT COMPLETE.")

AssertionError: 

In [6]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# --------------------------------------------------
# 1. Reproducibility and configuration
# --------------------------------------------------

SEED = 42
BATCH_SIZE = 4
NUM_WORKERS = 4

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

# --------------------------------------------------
# 2. Load datasets
# --------------------------------------------------

base_dir = Path("./sentinel1-lulc")

train_path = base_dir / "patches_v5" / "train_v5.npz"
val_path = base_dir / "patches_v4" / "val_v4.npz"
test_path = base_dir / "patches_v4" / "test_v4.npz"

def load_dataset(path):
    with np.load(path, allow_pickle=False) as data:
        X = data["X"].copy()
        y = data["y"].copy()
        medians = data["class_medians"].copy()
        iqr = data["class_iqr"].copy()

    return X, y, medians, iqr

X_train, y_train, train_medians, train_iqr = load_dataset(train_path)
X_val, y_val, val_medians, val_iqr = load_dataset(val_path)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

# --------------------------------------------------
# 3. Verify normalization consistency
# --------------------------------------------------

assert np.allclose(train_medians, val_medians, atol=1e-5)
assert np.allclose(train_iqr, val_iqr, atol=1e-5)

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()

assert set(np.unique(y_train)).issubset({1, 2, 3, 4, 5, 6, 255})
assert set(np.unique(y_val)).issubset({1, 2, 3, 4, 5, 6, 255})

print("\nNormalization and label checks passed.")

# --------------------------------------------------
# 4. Convert to PyTorch tensors
# --------------------------------------------------

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train.astype(np.int64))
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val.astype(np.int64))
)

# --------------------------------------------------
# 5. V4-compatible weighted sampler
# --------------------------------------------------

bare_counts = np.count_nonzero(y_train == 5, axis=(1, 2))
water_counts = np.count_nonzero(y_train == 6, axis=(1, 2))

rare_patch = (bare_counts >= 100) | (water_counts >= 100)

sample_weights = np.where(
    rare_patch, 2.0, 1.0
).astype(np.float64)

sampler_generator = torch.Generator()
sampler_generator.manual_seed(SEED)

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
    generator=sampler_generator
)

print("\nSampler distribution:")
print("Weight 1:", int(np.sum(sample_weights == 1)))
print("Weight 2:", int(np.sum(sample_weights == 2)))

# --------------------------------------------------
# 6. Create DataLoaders
# --------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

# --------------------------------------------------
# 7. Inspect one batch
# --------------------------------------------------

xb, yb = next(iter(train_loader))

print("\nBATCH TEST")
print("Input:", xb.shape, xb.dtype)
print("Labels:", yb.shape, yb.dtype)
print("Input device before transfer:", xb.device)
print("Label values:", torch.unique(yb).tolist())

assert xb.shape == (BATCH_SIZE, 2, 128, 128)
assert yb.shape == (BATCH_SIZE, 128, 128)

print("\nV5 DATALOADERS READY.")

Training device: cuda


FileNotFoundError: [Errno 2] No such file or directory: 'sentinel1-lulc/patches_v5/train_v5.npz'

In [7]:
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Double Convolution Block
# --------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels, out_channels,
                kernel_size=3, padding=1, bias=False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# --------------------------------------------------
# 2. U-Net Architecture
# --------------------------------------------------

class UNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=6, base_channels=16):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.enc4 = DoubleConv(base_channels * 4, base_channels * 8)
        self.bottleneck = DoubleConv(base_channels * 8, base_channels * 16)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Decoder
        self.up4 = nn.ConvTranspose2d(
            base_channels * 16, base_channels * 8,
            kernel_size=2, stride=2
        )
        self.dec4 = DoubleConv(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(
            base_channels * 8, base_channels * 4,
            kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2,
            kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels,
            kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        # Output
        self.out = nn.Conv2d(
            base_channels, num_classes, kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        # Decoder with skip connections
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


# --------------------------------------------------
# 3. Instantiate V5 model
# --------------------------------------------------

model = UNet(
    in_channels=2,
    num_classes=6,
    base_channels=16
).to(device)

# --------------------------------------------------
# 4. Architecture verification
# --------------------------------------------------

model.eval()

with torch.no_grad():
    dummy_input = torch.randn(
        1, 2, 128, 128,
        device=device
    )

    dummy_output = model(dummy_input)

print("INPUT SHAPE:", dummy_input.shape)
print("OUTPUT SHAPE:", dummy_output.shape)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

assert dummy_output.shape == (1, 6, 128, 128)
assert torch.isfinite(dummy_output).all()

print("\nV5 U-NET ARCHITECTURE VERIFIED.")

INPUT SHAPE: torch.Size([1, 2, 128, 128])
OUTPUT SHAPE: torch.Size([1, 6, 128, 128])
Total parameters: 1,942,518
Trainable parameters: 1,942,518

V5 U-NET ARCHITECTURE VERIFIED.


In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW

from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# --------------------------------------------------
# 1. Convert labels from 1-6 to 0-5
# --------------------------------------------------

def convert_labels(y):
    y = y.astype(np.int64)

    converted = np.where(
        y == 255,
        255,
        y - 1
    )

    return converted.astype(np.int64)


y_train_converted = convert_labels(y_train)
y_val_converted = convert_labels(y_val)

assert set(np.unique(y_train_converted)).issubset(
    {0, 1, 2, 3, 4, 5, 255}
)

assert set(np.unique(y_val_converted)).issubset(
    {0, 1, 2, 3, 4, 5, 255}
)

print("Training labels:", np.unique(y_train_converted))
print("Validation labels:", np.unique(y_val_converted))

# --------------------------------------------------
# 2. Recreate datasets with corrected labels
# --------------------------------------------------

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train_converted)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(y_val_converted)
)

# --------------------------------------------------
# 3. Recreate sampler using original labels
# --------------------------------------------------

bare_counts = np.count_nonzero(y_train == 5, axis=(1, 2))
water_counts = np.count_nonzero(y_train == 6, axis=(1, 2))

rare_patch = (bare_counts >= 100) | (water_counts >= 100)

sample_weights = np.where(
    rare_patch, 2.0, 1.0
).astype(np.float64)

sampler_generator = torch.Generator()
sampler_generator.manual_seed(42)

train_sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_dataset),
    replacement=True,
    generator=sampler_generator
)

# --------------------------------------------------
# 4. Recreate DataLoaders
# --------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=train_sampler,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# --------------------------------------------------
# 5. Configure weighted CrossEntropy loss
# --------------------------------------------------

class_weights = torch.tensor(
    [0.48, 0.48, 0.48, 0.48, 2.398, 1.684],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=255
)

# --------------------------------------------------
# 6. Configure optimizer
# --------------------------------------------------

optimizer = AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# --------------------------------------------------
# 7. Verify one batch and loss
# --------------------------------------------------

xb, yb = next(iter(train_loader))

xb = xb.to(device, non_blocking=True)
yb = yb.to(device, non_blocking=True)

model.train()

logits = model(xb)
loss = criterion(logits, yb)

assert logits.shape == (4, 6, 128, 128)
assert torch.isfinite(loss)

print("\nTRAINING CONFIGURATION")
print("Device:", device)
print("Batch size:", 4)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
print("Class weights:", class_weights.tolist())
print("Ignore index:", criterion.ignore_index)
print("Test batch loss:", round(loss.item(), 5))

print("\nLABEL CONVERSION AND TRAINING CONFIGURATION VERIFIED.")


NameError: name 'y_train' is not defined

In [9]:
import json
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Training configuration
# --------------------------------------------------

EPOCHS = 30
BEST_VAL_MIOU = -1.0

output_dir = Path("./sentinel1-lulc/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = output_dir / "best_unet_v5.pt"
history_path = output_dir / "training_history_v5.json"

history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_miou": [],
    "epoch_seconds": []
}

# --------------------------------------------------
# 2. Validation metrics
# --------------------------------------------------

@torch.no_grad()
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    total_batches = 0
    confusion = torch.zeros(
        (6, 6), dtype=torch.int64, device=device
    )

    for xb, yb in loader:

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item()
        total_batches += 1

        predictions = logits.argmax(dim=1)

        valid = yb != 255

        target_valid = yb[valid]
        pred_valid = predictions[valid]

        encoded = target_valid * 6 + pred_valid

        confusion += torch.bincount(
            encoded,
            minlength=36
        ).reshape(6, 6)

    val_loss = total_loss / total_batches

    true_positive = confusion.diag().float()
    actual = confusion.sum(dim=1).float()
    predicted = confusion.sum(dim=0).float()

    union = actual + predicted - true_positive

    iou = torch.where(
        union > 0,
        true_positive / union,
        torch.zeros_like(union)
    )

    accuracy = (
        true_positive.sum() / actual.sum()
    ).item()

    mean_iou = iou.mean().item()

    return (
        val_loss,
        accuracy,
        mean_iou,
        iou.cpu().numpy(),
        confusion.cpu().numpy()
    )

# --------------------------------------------------
# 3. Training loop
# --------------------------------------------------

print("=" * 65)
print("STARTING V5 TRAINING")
print("=" * 65)

print("Training observations:", len(train_dataset))
print("Validation observations:", len(val_dataset))
print("Batches per epoch:", len(train_loader))
print("Maximum epochs:", EPOCHS)
print("Device:", device)

training_start = time.time()

for epoch in range(EPOCHS):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    batches = 0

    for xb, yb in train_loader:

        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batches += 1

    train_loss = running_loss / batches

    # Validation
    (
        val_loss,
        val_accuracy,
        val_miou,
        val_iou,
        val_confusion
    ) = evaluate(model, val_loader, criterion, device)

    epoch_seconds = time.time() - epoch_start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_miou"].append(val_miou)
    history["epoch_seconds"].append(epoch_seconds)

    improved = val_miou > BEST_VAL_MIOU

    if improved:

        BEST_VAL_MIOU = val_miou

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "best_val_miou": BEST_VAL_MIOU,
            "architecture": "UNet",
            "in_channels": 2,
            "num_classes": 6,
            "base_channels": 16,
            "class_medians": [-9.042964, -15.390633],
            "class_iqr": [3.967108, 4.4696884],
            "seed": 42
        }, checkpoint_path)

    # Save history after every epoch.
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val mIoU: {val_miou:.4f} | "
        f"Best: {BEST_VAL_MIOU:.4f} | "
        f"Time: {epoch_seconds / 60:.1f} min"
        f"{'  <-- BEST' if improved else ''}"
    )

    # Save current epoch validation diagnostics.
    if improved:
        np.save(
            output_dir / "validation_iou_v5.npy",
            val_iou
        )
        np.save(
            output_dir / "validation_confusion_v5.npy",
            val_confusion
        )

total_minutes = (time.time() - training_start) / 60

print("\n" + "=" * 65)
print("V5 TRAINING FINISHED")
print("=" * 65)
print("Best validation mIoU:", round(BEST_VAL_MIOU, 4))
print("Checkpoint:", checkpoint_path)
print("Training history:", history_path)
print("Total training time:", round(total_minutes, 1), "minutes")


STARTING V5 TRAINING


NameError: name 'train_dataset' is not defined

In [6]:
import numpy as np
import torch
from pathlib import Path

output_dir = Path("./sentinel1-lulc/outputs")

# Load saved validation metrics
iou = np.load(output_dir / "validation_iou_v5.npy")
cm = np.load(output_dir / "validation_confusion_v5.npy")

class_names = [
    "Tree cover",
    "Grassland",
    "Cropland",
    "Built-up",
    "Bare/sparse vegetation",
    "Permanent water"
]

print("=" * 65)
print("V5 BEST-CHECKPOINT VALIDATION RESULTS")
print("=" * 65)

print(f"\nPer-class IoU (mean IoU: {np.nanmean(iou):.4f})")
for name, score in zip(class_names, iou):
    print(f"{name:25s}: {score:.4f}")

# Confusion matrix: rows = actual, columns = predicted
print("\nPer-class precision and recall")

for i, name in enumerate(class_names):
    tp = cm[i, i]
    actual = cm[i, :].sum()
    predicted = cm[:, i].sum()

    recall = tp / actual if actual > 0 else np.nan
    precision = tp / predicted if predicted > 0 else np.nan

    print(
        f"{name:25s} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )

print("\nConfusion matrix:")
print(cm)

# Verify checkpoint metadata
checkpoint = torch.load(
    output_dir / "best_unet_v5.pt",
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint verification")
print("Best epoch:", checkpoint["epoch"])
print("Best validation mIoU:", checkpoint["best_val_miou"])
print("Architecture:", checkpoint["architecture"])
print("Number of classes:", checkpoint["num_classes"])

V5 BEST-CHECKPOINT VALIDATION RESULTS

Per-class IoU (mean IoU: 0.5445)
Tree cover               : 0.8301
Grassland                : 0.3871
Cropland                 : 0.7442
Built-up                 : 0.6077
Bare/sparse vegetation   : 0.0000
Permanent water          : 0.6979

Per-class precision and recall
Tree cover                | Precision: 0.9300 | Recall: 0.8855
Grassland                 | Precision: 0.6489 | Recall: 0.4896
Cropland                  | Precision: 0.8299 | Recall: 0.8782
Built-up                  | Precision: 0.6777 | Recall: 0.8548
Bare/sparse vegetation    | Precision: 0.0000 | Recall: 0.0000
Permanent water           | Precision: 0.7590 | Recall: 0.8965

Confusion matrix:
[[2886331   46189   28412  293883       0    4805]
 [  99212  216255   93791   31742       2     706]
 [  24601   56698  663610   10703      26      26]
 [  91918   13756   13769  708270       8     891]
 [     23      18      16     171       0       0]
 [   1629     332       3     373       

In [2]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path

# --------------------------------------------------
# 1. Load the best V5 checkpoint
# --------------------------------------------------
output_dir = Path("./sentinel1-lulc/outputs")
test_path = Path("./sentinel1-lulc/patches_v4/test_v4.npz")

checkpoint = torch.load(
    output_dir / "best_unet_v5.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Loaded V5 checkpoint")
print("Best epoch:", checkpoint["epoch"])
print("Best validation mIoU:", checkpoint["best_val_miou"])

# --------------------------------------------------
# 2. Load the unchanged V4 test observations
# --------------------------------------------------
test_data = np.load(test_path)

X_test = test_data["X"].astype(np.float32)
y_test_original = test_data["y"].astype(np.uint8)

# Convert class IDs 1..6 to 0..5; preserve ignore=255
y_test = np.where(
    y_test_original == 255,
    255,
    y_test_original - 1
).astype(np.uint8)

assert X_test.shape == (324, 2, 128, 128)
assert y_test.shape == (324, 128, 128)
assert np.isfinite(X_test).all()
assert set(np.unique(y_test)).issubset({0, 1, 2, 3, 4, 5, 255})

test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test.astype(np.int64))
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# --------------------------------------------------
# 3. Evaluate exactly once on the test set
# --------------------------------------------------
num_classes = 6
confusion = np.zeros((num_classes, num_classes), dtype=np.int64)

total_loss = 0.0
total_batches = 0

model.eval()

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, targets)

        total_loss += loss.item()
        total_batches += 1

        predictions = logits.argmax(dim=1)

        valid = targets != 255
        true_valid = targets[valid].cpu().numpy()
        pred_valid = predictions[valid].cpu().numpy()

        encoded = true_valid * num_classes + pred_valid
        batch_confusion = np.bincount(
            encoded,
            minlength=num_classes * num_classes
        ).reshape(num_classes, num_classes)

        confusion += batch_confusion

# --------------------------------------------------
# 4. Calculate aggregate metrics
# --------------------------------------------------
tp = np.diag(confusion).astype(np.float64)
actual = confusion.sum(axis=1).astype(np.float64)
predicted = confusion.sum(axis=0).astype(np.float64)

union = actual + predicted - tp

iou = np.divide(
    tp, union,
    out=np.zeros_like(tp),
    where=union > 0
)

precision = np.divide(
    tp, predicted,
    out=np.zeros_like(tp),
    where=predicted > 0
)

recall = np.divide(
    tp, actual,
    out=np.zeros_like(tp),
    where=actual > 0
)

accuracy = tp.sum() / actual.sum()
mean_iou = iou.mean()
test_loss = total_loss / total_batches

class_names = [
    "Tree cover",
    "Grassland",
    "Cropland",
    "Built-up",
    "Bare/sparse vegetation",
    "Permanent water"
]

# --------------------------------------------------
# 5. Print results
# --------------------------------------------------
print("\n" + "=" * 65)
print("V5 HELD-OUT TEST RESULTS")
print("=" * 65)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Test mIoU:     {mean_iou:.4f}")

print("\nPer-class metrics:")
for i, name in enumerate(class_names):
    print(
        f"{name:25s} | "
        f"IoU: {iou[i]:.4f} | "
        f"Precision: {precision[i]:.4f} | "
        f"Recall: {recall[i]:.4f} | "
        f"Pixels: {int(actual[i]):,}"
    )

print("\nConfusion matrix:")
print(confusion)

# Save test results for project records
np.save(output_dir / "test_confusion_v5.npy", confusion)
np.save(output_dir / "test_iou_v5.npy", iou)

print("\nSaved:")
print(output_dir / "test_confusion_v5.npy")
print(output_dir / "test_iou_v5.npy")

NameError: name 'device' is not defined

In [1]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Test metadata contains the month for each observation
metadata = test_data["metadata"]
months = sorted(np.unique(metadata["month"]).tolist())

monthly_results = []

model.eval()

for month in months:
    indices = np.where(metadata["month"] == month)[0]

    X_month = X_test[indices]
    y_month = y_test[indices]

    dataset = TensorDataset(
        torch.from_numpy(X_month),
        torch.from_numpy(y_month.astype(np.int64))
    )

    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0
    )

    cm = np.zeros((6, 6), dtype=np.int64)

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            predictions = model(images).argmax(dim=1)

            valid = targets != 255
            true_valid = targets[valid].cpu().numpy()
            pred_valid = predictions[valid].cpu().numpy()

            encoded = true_valid * 6 + pred_valid

            cm += np.bincount(
                encoded,
                minlength=36
            ).reshape(6, 6)

    tp = np.diag(cm).astype(float)
    actual = cm.sum(axis=1).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    union = actual + predicted - tp

    iou = np.divide(
        tp,
        union,
        out=np.zeros(6, dtype=float),
        where=union > 0
    )

    accuracy = tp.sum() / actual.sum()
    miou = iou.mean()

    monthly_results.append({
        "month": month,
        "observations": len(indices),
        "accuracy": accuracy,
        "mIoU": miou,
        "tree": iou[0],
        "grass": iou[1],
        "crop": iou[2],
        "built": iou[3],
        "bare": iou[4],
        "water": iou[5]
    })

print("=" * 100)
print("V5 MONTHLY HELD-OUT TEST PERFORMANCE")
print("=" * 100)

print(
    f"{'Month':10s} {'mIoU':>8s} {'Accuracy':>10s} "
    f"{'Tree':>8s} {'Grass':>8s} {'Crop':>8s} "
    f"{'Built':>8s} {'Bare':>8s} {'Water':>8s}"
)

for r in monthly_results:
    print(
        f"{r['month']:10s} "
        f"{r['mIoU']:8.4f} "
        f"{r['accuracy']:10.4f} "
        f"{r['tree']:8.4f} "
        f"{r['grass']:8.4f} "
        f"{r['crop']:8.4f} "
        f"{r['built']:8.4f} "
        f"{r['bare']:8.4f} "
        f"{r['water']:8.4f}"
    )

monthly_mious = [r["mIoU"] for r in monthly_results]

print("\nMonthly mIoU summary")
print(f"Mean:   {np.mean(monthly_mious):.4f}")
print(f"Std:    {np.std(monthly_mious):.4f}")
print(f"Min:    {np.min(monthly_mious):.4f}")
print(f"Max:    {np.max(monthly_mious):.4f}")

# Save the monthly breakdown
import json
from pathlib import Path

output_path = Path("./sentinel1-lulc/outputs/monthly_test_metrics_v5.json")

with open(output_path, "w") as f:
    json.dump(monthly_results, f, indent=2)

print("\nSaved:", output_path)

NameError: name 'test_data' is not defined

In [1]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nContents of current directory:")
for p in Path.cwd().iterdir():
    print(p.name, "/" if p.is_dir() else "")

print("\nChecking possible V5 dataset locations:")

candidates = [
    Path("./sentinel1-lulc/patches_v5/train_v5.npz"),
    Path("./patches_v5/train_v5.npz"),
    Path("../sentinel1-lulc/patches_v5/train_v5.npz"),
]

for p in candidates:
    print(f"{p} -> exists: {p.exists()}")

Current working directory:
/home/jovyan/vault/Project/DeepLearning/sentinel1-lulc

Contents of current directory:
stuttgart_worldcover_2021_utm.tif 
.ipynb_checkpoints /
stuttgart_s1_vv_vh_june2025_utm.tif 
stuttgart_worldcover_20km_2021.tif 
stuttgart_s1_vv_vh_20km_june2025.tif 
patches /
outputs /
monthly_inputs /
monthly_maps /
patches_v4 /
v5_data /
patches_v5_outer_ring /
patches_v5 /
09_train_unet_v5_expanded_area.ipynb 

Checking possible V5 dataset locations:
sentinel1-lulc/patches_v5/train_v5.npz -> exists: False
patches_v5/train_v5.npz -> exists: True
../sentinel1-lulc/patches_v5/train_v5.npz -> exists: True


In [2]:
from pathlib import Path

project_dir = Path.cwd()

v5_path = project_dir / "patches_v5" / "train_v5.npz"
v4_val_path = project_dir / "patches_v4" / "val_v4.npz"
v4_test_path = project_dir / "patches_v4" / "test_v4.npz"
output_dir = project_dir / "outputs"

assert v5_path.exists(), f"Missing: {v5_path}"
assert v4_val_path.exists(), f"Missing: {v4_val_path}"
assert v4_test_path.exists(), f"Missing: {v4_test_path}"
assert (output_dir / "best_unet_v5.pt").exists()

print("All required files found.")
print("V5 training dataset:", v5_path)
print("V4 validation dataset:", v4_val_path)
print("V4 test dataset:", v4_test_path)
print("V5 checkpoint:", output_dir / "best_unet_v5.pt")

All required files found.
V5 training dataset: /home/jovyan/vault/Project/DeepLearning/sentinel1-lulc/patches_v5/train_v5.npz
V4 validation dataset: /home/jovyan/vault/Project/DeepLearning/sentinel1-lulc/patches_v4/val_v4.npz
V4 test dataset: /home/jovyan/vault/Project/DeepLearning/sentinel1-lulc/patches_v4/test_v4.npz
V5 checkpoint: /home/jovyan/vault/Project/DeepLearning/sentinel1-lulc/outputs/best_unet_v5.pt


In [4]:
import torch
from pathlib import Path

output_dir = Path.cwd() / "outputs"
checkpoint_path = output_dir / "best_unet_v5.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

print("=" * 55)
print("V5 MODEL RESTORED")
print("=" * 55)

print("Checkpoint:", checkpoint_path)
print("Best epoch:", checkpoint["epoch"])
print("Best validation mIoU:", checkpoint["best_val_miou"])
print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

# Verify inference
with torch.no_grad():
    dummy = torch.zeros(1, 2, 128, 128, device=device)
    prediction = model(dummy)

assert prediction.shape == (1, 6, 128, 128)
assert torch.isfinite(prediction).all()

print("Inference verification: PASSED")

NameError: name 'device' is not defined

In [10]:
import torch
from pathlib import Path

checkpoint_path = Path.cwd() / "outputs" / "best_unet_v5.pt"

assert checkpoint_path.exists(), "V5 checkpoint not found."

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("V5 MODEL RESTORED")
print("Best epoch:", checkpoint["epoch"])
print("Best validation mIoU:", checkpoint["best_val_miou"])
print("Device:", device)

with torch.no_grad():
    dummy = torch.zeros(1, 2, 128, 128, device=device)
    prediction = model(dummy)

assert prediction.shape == (1, 6, 128, 128)
assert torch.isfinite(prediction).all()

print("Inference verification: PASSED")

V5 MODEL RESTORED
Best epoch: 6
Best validation mIoU: 0.5445035696029663
Device: cuda
Inference verification: PASSED


In [11]:
import numpy as np
import json
from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader

# --------------------------------------------------
# 1. Load original held-out test dataset
# --------------------------------------------------
project_dir = Path.cwd()
test_path = project_dir / "patches_v4" / "test_v4.npz"
output_dir = project_dir / "outputs"

assert test_path.exists(), f"Missing test data: {test_path}"
assert model is not None, "V5 model is not restored."

test_data = np.load(test_path)

X_test = test_data["X"].astype(np.float32)
y_original = test_data["y"].astype(np.uint8)
metadata = test_data["metadata"]

y_test = np.where(
    y_original == 255,
    255,
    y_original - 1
).astype(np.uint8)

assert X_test.shape == (324, 2, 128, 128)
assert y_test.shape == (324, 128, 128)
assert len(metadata) == 324

months = sorted(np.unique(metadata["month"]).tolist())

print("Test observations:", len(X_test))
print("Months:", months)

# --------------------------------------------------
# 2. Evaluate each month independently
# --------------------------------------------------
monthly_results = []
model.eval()

for month in months:
    indices = np.where(metadata["month"] == month)[0]

    dataset = TensorDataset(
        torch.from_numpy(X_test[indices]),
        torch.from_numpy(y_test[indices].astype(np.int64))
    )

    loader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0
    )

    cm = np.zeros((6, 6), dtype=np.int64)

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            predictions = model(images).argmax(dim=1)

            valid = targets != 255
            true_valid = targets[valid].cpu().numpy()
            pred_valid = predictions[valid].cpu().numpy()

            encoded = true_valid * 6 + pred_valid

            cm += np.bincount(
                encoded,
                minlength=36
            ).reshape(6, 6)

    tp = np.diag(cm).astype(float)
    actual = cm.sum(axis=1).astype(float)
    predicted = cm.sum(axis=0).astype(float)
    union = actual + predicted - tp

    iou = np.divide(
        tp,
        union,
        out=np.zeros(6),
        where=union > 0
    )

    accuracy = tp.sum() / actual.sum()
    miou = iou.mean()

    monthly_results.append({
        "month": month,
        "observations": int(len(indices)),
        "accuracy": float(accuracy),
        "mIoU": float(miou),
        "tree": float(iou[0]),
        "grass": float(iou[1]),
        "crop": float(iou[2]),
        "built": float(iou[3]),
        "bare": float(iou[4]),
        "water": float(iou[5])
    })

# --------------------------------------------------
# 3. Display results
# --------------------------------------------------
print("\n" + "=" * 105)
print("V5 MONTHLY HELD-OUT TEST PERFORMANCE")
print("=" * 105)

print(
    f"{'Month':10s} {'mIoU':>8s} {'Accuracy':>10s} "
    f"{'Tree':>8s} {'Grass':>8s} {'Crop':>8s} "
    f"{'Built':>8s} {'Bare':>8s} {'Water':>8s}"
)

for r in monthly_results:
    print(
        f"{r['month']:10s} "
        f"{r['mIoU']:8.4f} "
        f"{r['accuracy']:10.4f} "
        f"{r['tree']:8.4f} "
        f"{r['grass']:8.4f} "
        f"{r['crop']:8.4f} "
        f"{r['built']:8.4f} "
        f"{r['bare']:8.4f} "
        f"{r['water']:8.4f}"
    )

# --------------------------------------------------
# 4. Summary and save
# --------------------------------------------------
monthly_mious = [r["mIoU"] for r in monthly_results]

print("\nMonthly mIoU summary")
print(f"Mean: {np.mean(monthly_mious):.4f}")
print(f"Std:  {np.std(monthly_mious):.4f}")
print(f"Min:  {np.min(monthly_mious):.4f}")
print(f"Max:  {np.max(monthly_mious):.4f}")

output_path = output_dir / "monthly_test_metrics_v5.json"

with open(output_path, "w") as f:
    json.dump(monthly_results, f, indent=2)

print("\nSaved:", output_path)

Test observations: 324
Months: ['2025_10', '2025_11', '2025_12', '2026_01', '2026_02', '2026_03', '2026_04', '2026_05', '2026_06', '2026_07', '2026_08', '2026_09']

V5 MONTHLY HELD-OUT TEST PERFORMANCE
Month          mIoU   Accuracy     Tree    Grass     Crop    Built     Bare    Water
2025_10      0.4617     0.7102   0.5300   0.2671   0.6312   0.6355   0.0000   0.7063
2025_11      0.4773     0.7195   0.5324   0.3449   0.6607   0.6300   0.0000   0.6957
2025_12      0.4842     0.7255   0.5391   0.3532   0.6808   0.6307   0.0000   0.7010
2026_01      0.4818     0.7280   0.5477   0.3445   0.6891   0.6302   0.0000   0.6794
2026_02      0.4806     0.7269   0.5449   0.3471   0.6916   0.6258   0.0000   0.6742
2026_03      0.4827     0.7295   0.5492   0.3380   0.6972   0.6299   0.0000   0.6819
2026_04      0.4843     0.7259   0.5515   0.3332   0.6681   0.6361   0.0000   0.7172
2026_05      0.4631     0.7147   0.5492   0.2923   0.6301   0.6316   0.0000   0.6756
2026_06      0.4551     0.7038   